In [2]:
from dataclasses import dataclass, field
from datetime import datetime


@dataclass
class Item:
    item_id: int
    title: str
    category: str
    community: str
    owner: str
    mode: str          # "Rent" or "Lend"
    rate_per_day: int  # 0 for "Lend"
    available: bool = True
    borrower: str = ""


@dataclass
class LendLocal:
    items: list[Item] = field(default_factory=list)
    history: list[str] = field(default_factory=list)
    _next_id: int = 1

    def add_item(
        self,
        title: str,
        category: str,
        community: str,
        owner: str,
        mode: str,
        rate_per_day: int = 0,
    ) -> Item:
        mode = mode.capitalize()
        if mode not in ("Rent", "Lend"):
            raise ValueError("Mode must be 'Rent' or 'Lend'.")

        fee = int(rate_per_day) if mode == "Rent" else 0
        item = Item(
            item_id=self._next_id,
            title=title.strip(),
            category=category.strip(),
            community=community.strip(),
            owner=owner.strip(),
            mode=mode,
            rate_per_day=fee,
        )
        self._next_id += 1
        self.items.append(item)
        self.history.append(
            f"[{self._now()}] LISTED: '{item.title}' ({item.mode}) in {item.community} by {item.owner}"
        )
        return item

    def show_items(self, community_filter: str | None = None, mode_filter: str | None = None) -> None:
        header = community_filter if community_filter else "All Community Regions"
        print(f"\n--- {header} Listings ---")
        found = False
        for item in self.items:
            if community_filter and item.community.lower() != community_filter.lower():
                continue
            if mode_filter and item.mode.lower() != mode_filter.lower():
                continue
            found = True
            status = "Available" if item.available else f"With {item.borrower}"
            pricing = f"₹{item.rate_per_day}/day" if item.mode == "Rent" else "Free (Lend)"
            print(
                f"#{item.item_id} | [{item.community}] {item.title} ({item.category}) | "
                f"Mode: {item.mode} [{pricing}] | Owner: {item.owner} | Status: {status}"
            )
        if not found:
            print("No items found.")

    def request_item(self, item_id: int, borrower: str, days: int = 1) -> None:
        if days < 1:
            print("Duration must be at least 1 day.")
            return

        item = self._find_item(item_id)
        if not item:
            print(f"Item #{item_id} not found.")
            return
        if not item.available:
            print(f"'{item.title}' is currently unavailable (with {item.borrower}).")
            return

        total_cost = item.rate_per_day * days if item.mode == "Rent" else 0
        item.available = False
        item.borrower = borrower.strip()

        action = "RENTED" if item.mode == "Rent" else "BORROWED"
        msg = (
            f"[{self._now()}] {action}: {item.borrower} took '{item.title}' "
            f"from {item.owner} ({item.community}) for {days} day(s) | Total: ₹{total_cost}"
        )
        self.history.append(msg)
        print(msg)

    def return_item(self, item_id: int) -> None:
        item = self._find_item(item_id)
        if not item:
            print(f"Item #{item_id} not found.")
            return
        if item.available:
            print(f"'{item.title}' is already available.")
            return

        previous_borrower = item.borrower
        item.available = True
        item.borrower = ""

        msg = (
            f"[{self._now()}] RETURNED: '{item.title}' returned by {previous_borrower} "
            f"to {item.owner} ({item.community})"
        )
        self.history.append(msg)
        print(msg)

    def show_history(self) -> None:
        print("\n--- Community Activity Log ---")
        if not self.history:
            print("No activity yet.")
            return
        for entry in self.history:
            print(entry)

    def _find_item(self, item_id: int) -> Item | None:
        for item in self.items:
            if item.item_id == item_id:
                return item
        return None

    @staticmethod
    def _now() -> str:
        return datetime.now().strftime("%Y-%m-%d %H:%M")


def load_default_data(app: LendLocal) -> None:
    # Original place: Kothrud West + 3 initial items
    app.add_item("Cordless Drill", "Tools", "Kothrud West", "Rohan", "Rent", rate_per_day=150)
    app.add_item("Python & Math Textbooks", "Books", "Kothrud West", "Ananya", "Lend")
    app.add_item("Camera Tripod", "Electronics", "Kothrud West", "Kabir", "Rent", rate_per_day=120)

    # First batch of 10 additional items (Kothrud West)
    app.add_item("Soldering Iron Kit", "Tools", "Kothrud West", "Aarav", "Rent", rate_per_day=80)
    app.add_item("Scientific Calculator", "Electronics", "Kothrud West", "Meera", "Lend")
    app.add_item("Badminton Racquet Pair", "Sports", "Kothrud West", "Siddharth", "Lend")
    app.add_item("2-Person Camping Tent", "Outdoors", "Kothrud West", "Neha", "Rent", rate_per_day=250)
    app.add_item("Extension Cord (10m)", "Household", "Kothrud West", "Vikram", "Lend")
    app.add_item("Measuring Tape & Spirit Level", "Tools", "Kothrud West", "Pooja", "Lend")
    app.add_item("Portable Bluetooth Speaker", "Electronics", "Kothrud West", "Dev", "Rent", rate_per_day=100)
    app.add_item("Board Game Collection", "Recreation", "Kothrud West", "Rhea", "Lend")
    app.add_item("Pressure Washer", "Household", "Kothrud West", "Kunal", "Rent", rate_per_day=300)
    app.add_item("Study Desk Lamp", "Household", "Kothrud West", "Tanvi", "Lend")

    # 3 more places + 10 more items
    # Place 2: Paud Road (4 items)
    app.add_item("Ring Light & Mic Set", "Electronics", "Paud Road", "Aditya", "Rent", rate_per_day=140)
    app.add_item("DSA & Algorithms Notes", "Books", "Paud Road", "Sneha", "Lend")
    app.add_item("Folding Step Ladder", "Tools", "Paud Road", "Manav", "Rent", rate_per_day=110)
    app.add_item("Yoga Mat & Block Set", "Sports", "Paud Road", "Ishita", "Lend")

    # Place 3: Bavdhan (3 items)
    app.add_item("Portable Projector", "Electronics", "Bavdhan", "Nikhil", "Rent", rate_per_day=350)
    app.add_item("Trekking Backpack (45L)", "Outdoors", "Bavdhan", "Kavya", "Lend")
    app.add_item("Electric Glue Gun", "Tools", "Bavdhan", "Harsh", "Lend")

    # Place 4: Karve Nagar (3 items)
    app.add_item("Sewing Machine", "Household", "Karve Nagar", "Sunita", "Rent", rate_per_day=130)
    app.add_item("Cricket Bat & Stumps", "Sports", "Karve Nagar", "Pranav", "Lend")
    app.add_item("Linear Algebra Workbook", "Books", "Karve Nagar", "Rutuja", "Lend")


def run_demo() -> None:
    app = LendLocal()
    load_default_data(app)

    app.show_items()

    # Sample rental and lending across different places
    app.request_item(item_id=1, borrower="Aarav", days=2)
    app.request_item(item_id=14, borrower="Rhea", days=3)
    app.request_item(item_id=18, borrower="Meera", days=4)

    app.show_items(community_filter="Paud Road")
    app.show_history()


if __name__ == "__main__":
    run_demo()


--- All Community Regions Listings ---
#1 | [Kothrud West] Cordless Drill (Tools) | Mode: Rent [₹150/day] | Owner: Rohan | Status: Available
#2 | [Kothrud West] Python & Math Textbooks (Books) | Mode: Lend [Free (Lend)] | Owner: Ananya | Status: Available
#3 | [Kothrud West] Camera Tripod (Electronics) | Mode: Rent [₹120/day] | Owner: Kabir | Status: Available
#4 | [Kothrud West] Soldering Iron Kit (Tools) | Mode: Rent [₹80/day] | Owner: Aarav | Status: Available
#5 | [Kothrud West] Scientific Calculator (Electronics) | Mode: Lend [Free (Lend)] | Owner: Meera | Status: Available
#6 | [Kothrud West] Badminton Racquet Pair (Sports) | Mode: Lend [Free (Lend)] | Owner: Siddharth | Status: Available
#7 | [Kothrud West] 2-Person Camping Tent (Outdoors) | Mode: Rent [₹250/day] | Owner: Neha | Status: Available
#8 | [Kothrud West] Extension Cord (10m) (Household) | Mode: Lend [Free (Lend)] | Owner: Vikram | Status: Available
#9 | [Kothrud West] Measuring Tape & Spirit Level (Tools) | Mode: Le